# 🎬 Sistema de Recomendación de Películas - MovieLens 100K

## Introducción a Sistemas de Recomendación con Collaborative Filtering

Este notebook educativo explora cómo construir un sistema de recomendación de películas usando múltiples técnicas de **Collaborative Filtering**.

### Objetivos del Notebook:
1. **Cargar y explorar** el dataset MovieLens 100K
2. **Analizar la esparsidad** de la matriz de ratings
3. **Implementar User-Based Collaborative Filtering** (similaridad entre usuarios)
4. **Implementar Item-Based Collaborative Filtering** (similaridad entre películas)
5. **Usar Matrix Factorization (SVD)** para predicciones latentes
6. **Crear un sistema híbrido** combinando múltiples enfoques

### Dataset: MovieLens 100K
- **943 usuarios** con historiales de calificaciones
- **1,682 películas** con metadatos
- **100,000 calificaciones** (escala 1-5 estrellas)
- **Sparsidad: 93.7%** - Solo el 6.3% de interacciones posibles existen

### El Desafío Principal:
¿Cómo **predecir qué películas le gustarán** a un usuario si la mayoría no las ha visto?

## Parte 1: Carga y Exploración de Datos

Comenzamos cargando el dataset MovieLens 100K y explorando su estructura básica.

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import urllib.request
import zipfile
from pathlib import Path
import shutil

# Configuración de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
%matplotlib inline

print("✅ Librerías importadas correctamente")


print("=" * 70)
print("📥 DESCARGANDO DATASET MOVIELENS 100K")
print("=" * 70)

# Configurar directorios
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
DATA_RAW_DIR = PROJECT_DIR / 'data' / 'raw'
ML100K_DIR = DATA_RAW_DIR / 'ml-100k'

print(f"\n📁 Proyecto: {PROJECT_DIR}")
print(f"📁 Datos: {ML100K_DIR}")

# Crear directorios
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Verificar si ya existe
if (ML100K_DIR / 'u.data').exists():
    print(f"\n✅ Dataset ya existe en: {ML100K_DIR}")
    print("   No es necesario descargar nuevamente.")
else:
    print(f"\n⏳ Dataset no encontrado. Iniciando descarga...\n")

    URL = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
    ZIP_FILE = DATA_RAW_DIR / "ml-100k.zip"

    try:
        # Descargar
        print(f"⏳ Descargando de: {URL}")
        print("   (Tamaño: ~5MB, puede tardar 30-60 segundos)\n")

        class DownloadProgress:
            def __init__(self):
                self.last_percent = 0

            def __call__(self, blocknum, blocksize, totalsize):
                downloaded = blocknum * blocksize
                percent = min(int((downloaded / totalsize) * 100), 100)
                if percent != self.last_percent and percent % 10 == 0:
                    print(f"   {percent}% descargado...")
                self.last_percent = percent

        urllib.request.urlretrieve(URL, ZIP_FILE, DownloadProgress())
        print("   ✅ Descarga completada\n")

        # Extraer
        print("⏳ Extrayendo archivos...")
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extractall(DATA_RAW_DIR)
        print("   ✅ Extracción completada\n")

        # Reorganizar archivos
        temp_dir = DATA_RAW_DIR / 'ml-100k'
        if temp_dir.exists() and temp_dir != ML100K_DIR:
            # Mover archivos al directorio correcto
            for file in temp_dir.glob('*'):
                dest = ML100K_DIR / file.name
                file.rename(dest)
            temp_dir.rmdir()

        # Limpiar ZIP
        ZIP_FILE.unlink()
        print(f"✅ Dataset listo en: {ML100K_DIR}\n")

    except Exception as e:
        print(f"\n❌ Error durante descarga: {e}")
        print("\n💡 Alternativas:")
        print("   1. Intenta descargar manualmente desde:")
        print("      http://files.grouplens.org/datasets/movielens/ml-100k.zip")
        print("   2. Extrae el ZIP en: amazone/data/raw/ml-100k/")
        print("   3. Vuelve a ejecutar esta celda")

print("\n" + "=" * 70)
print("📋 Verificando archivos...")
print("=" * 70)

required_files = ['u.data', 'u.item', 'u.user', 'u.genre']
all_exists = True

for file in required_files:
    file_path = ML100K_DIR / file
    exists = file_path.exists()
    symbol = "✅" if exists else "❌"
    print(f"{symbol} {file}")
    all_exists = all_exists and exists

print("\n" + "=" * 70)
if all_exists:
    print("✅ ¡Dataset listo para usar!")
else:
    print("❌ Algunos archivos faltan. Intenta descargar manualmente.")
print("=" * 70)

✅ Librerías importadas correctamente
📥 DESCARGANDO DATASET MOVIELENS 100K

📁 Proyecto: /
📁 Datos: /data/raw/ml-100k

⏳ Dataset no encontrado. Iniciando descarga...

⏳ Descargando de: http://files.grouplens.org/datasets/movielens/ml-100k.zip
   (Tamaño: ~5MB, puede tardar 30-60 segundos)

   10% descargado...
   20% descargado...
   30% descargado...
   40% descargado...
   50% descargado...
   60% descargado...
   70% descargado...
   80% descargado...
   90% descargado...
   100% descargado...
   ✅ Descarga completada

⏳ Extrayendo archivos...
   ✅ Extracción completada

✅ Dataset listo en: /data/raw/ml-100k


📋 Verificando archivos...
✅ u.data
✅ u.item
✅ u.user
✅ u.genre

✅ ¡Dataset listo para usar!

🚀 Próximo paso: Ejecuta el notebook principal
   amazone_sistema_recomendacion.ipynb


### Configuración de Rutas

Definimos las rutas necesarias para acceder a los datos. El dataset debe estar en `data/raw/ml-100k/`.

Si aún no tienes descargado el dataset, ejecuta primero:
```bash
python download_movielens.py
```

In [5]:
# Configurar rutas
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_DIR / 'data' / 'raw' / 'ml-100k'
REPORTS_DIR = PROJECT_DIR / 'reports'

# Crear directorio de reportes si no existe
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Rutas configuradas:")
print(f"   - Proyecto: {PROJECT_DIR}")
print(f"   - Datos: {DATA_DIR}")
print(f"   - Reportes: {REPORTS_DIR}")
print(f"\n✅ Dataset disponible: {DATA_DIR.exists()}")

📁 Rutas configuradas:
   - Proyecto: /
   - Datos: /data/raw/ml-100k
   - Reportes: /reports

✅ Dataset disponible: True


### Cargar los Datos

El dataset MovieLens 100K contiene varios archivos:
- **u.data**: Ratings de usuarios (user_id, item_id, rating, timestamp)
- **u.item**: Información de películas (id, título, fecha de lanzamiento, géneros)
- **u.user**: Información demográfica de usuarios (id, edad, sexo, ocupación)

In [6]:
# Cargar ratings (calificaciones)
# Formato: user_id | item_id | rating | timestamp
ratings = pd.read_csv(
    DATA_DIR / 'u.data',
    sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp']
)

# Cargar información de películas
# Formato: movie_id | title | release_date | ... | géneros
movies = pd.read_csv(
    DATA_DIR / 'u.item',
    sep='|',
    encoding='latin-1',
    names=['item_id', 'title', 'release_date', 'video_release_date', 'imdb_url',
           'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
           'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
           'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
)

# Cargar información de usuarios
users = pd.read_csv(
    DATA_DIR / 'u.user',
    sep='|',
    names=['user_id', 'age', 'gender', 'occupation', 'zip_code']
)

print("✅ Datos cargados exitosamente!")

✅ Datos cargados exitosamente!


### Exploración Inicial del Dataset

In [7]:
print("=" * 60)
print("📊 INFORMACIÓN BÁSICA DEL DATASET")
print("=" * 60)

print(f"\n🎬 Películas: {movies.shape[0]:,}")
print(f"👥 Usuarios: {users.shape[0]:,}")
print(f"⭐ Ratings totales: {ratings.shape[0]:,}")

print("\n📊 PRIMERAS FILAS DE RATINGS:")
print(ratings.head())

print("\n📊 INFORMACIÓN DE RATINGS:")
print(ratings.info())

print("\n📈 ESTADÍSTICAS DE RATINGS:")
print(ratings.describe())

📊 INFORMACIÓN BÁSICA DEL DATASET

🎬 Películas: 1,682
👥 Usuarios: 943
⭐ Ratings totales: 100,000

📊 PRIMERAS FILAS DE RATINGS:
   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596

📊 INFORMACIÓN DE RATINGS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   item_id    100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB
None

📈 ESTADÍSTICAS DE RATINGS:
            user_id        item_id         rating     timestamp
count  100000.00000  100000.000000  100000.000000  1.000000e+05
mean      462.48475     425.530130       3.529860  8.835289e+08
std    

### Análisis de Ratings

In [8]:
print("\n" + "=" * 60)
print("⭐ ANÁLISIS DE CALIFICACIONES")
print("=" * 60)

# Distribución de ratings
rating_counts = ratings['rating'].value_counts().sort_index()
print("\n📊 Distribución de calificaciones:")
for rating, count in rating_counts.items():
    percentage = (count / len(ratings)) * 100
    bar = '█' * int(percentage / 2)
    print(f"Rating {rating}: {count:,} ({percentage:.1f}%) {bar}")

# Rating promedio
avg_rating = ratings['rating'].mean()
print(f"\n📊 Rating promedio: {avg_rating:.2f}")


⭐ ANÁLISIS DE CALIFICACIONES

📊 Distribución de calificaciones:
Rating 1: 6,110 (6.1%) ███
Rating 2: 11,370 (11.4%) █████
Rating 3: 27,145 (27.1%) █████████████
Rating 4: 34,174 (34.2%) █████████████████
Rating 5: 21,201 (21.2%) ██████████

📊 Rating promedio: 3.53


### Análisis de Usuarios y Películas

In [ ]:
print("\n" + "=" * 60)
print("👥 ANÁLISIS DE USUARIOS")
print("=" * 60)

# Ratings por usuario
ratings_per_user = ratings.groupby('user_id').size()
print(f"\n📊 Ratings por usuario:")
print(f"  - Promedio: {ratings_per_user.mean():.1f}")
print(f"  - Mínimo: {ratings_per_user.min()}")
print(f"  - Máximo: {ratings_per_user.max()}")
print(f"  - Mediana: {ratings_per_user.median():.1f}")

# Usuario más activo
most_active_user = ratings_per_user.idxmax()
print(f"\n🏆 Usuario más activo: {most_active_user} con {ratings_per_user.max()} ratings")

print("\n" + "=" * 60)
print("🎬 ANÁLISIS DE PELÍCULAS")
print("=" * 60)

# Ratings por película
ratings_per_movie = ratings.groupby('item_id').size()
print(f"\n📊 Ratings por película:")
print(f"  - Promedio: {ratings_per_movie.mean():.1f}")
print(f"  - Mínimo: {ratings_per_movie.min()}")
print(f"  - Máximo: {ratings_per_movie.max()}")
print(f"  - Mediana: {ratings_per_movie.median():.1f}")

# Películas más calificadas
most_rated_movies = ratings.groupby('item_id').size().sort_values(ascending=False).head(10)
print(f"\n🏆 TOP 10 PELÍCULAS MÁS CALIFICADAS:")
for idx, (movie_id, count) in enumerate(most_rated_movies.items(), 1):
    movie_title = movies[movies['item_id'] == movie_id]['title'].values[0]
    print(f"{idx:2d}. {movie_title}: {count} ratings")

### Visualizaciones Exploratorias

In [ ]:
# Crear figura con subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Análisis Exploratorio - MovieLens 100K', fontsize=16, fontweight='bold')

# 1. Distribución de ratings
axes[0, 0].bar(rating_counts.index, rating_counts.values, color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].set_title('Distribución de Calificaciones')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Distribución de ratings por usuario
axes[0, 1].hist(ratings_per_user, bins=50, color='coral', edgecolor='black')
axes[0, 1].set_xlabel('Número de ratings')
axes[0, 1].set_ylabel('Número de usuarios')
axes[0, 1].set_title('Distribución de Ratings por Usuario')
axes[0, 1].axvline(ratings_per_user.mean(), color='red', linestyle='--', label=f'Media: {ratings_per_user.mean():.1f}')
axes[0, 1].legend()

# 3. Distribución de ratings por película
axes[1, 0].hist(ratings_per_movie, bins=50, color='lightgreen', edgecolor='black')
axes[1, 0].set_xlabel('Número de ratings')
axes[1, 0].set_ylabel('Número de películas')
axes[1, 0].set_title('Distribución de Ratings por Película')
axes[1, 0].axvline(ratings_per_movie.mean(), color='red', linestyle='--', label=f'Media: {ratings_per_movie.mean():.1f}')
axes[1, 0].legend()

# 4. Rating promedio por película (top 20 más calificadas)
top_20_movies = most_rated_movies.head(20).index
top_20_avg_rating = ratings[ratings['item_id'].isin(top_20_movies)].groupby('item_id')['rating'].mean().sort_values(ascending=False)
axes[1, 1].barh(range(len(top_20_avg_rating)), top_20_avg_rating.values, color='plum')
axes[1, 1].set_yticks(range(len(top_20_avg_rating)))
axes[1, 1].set_yticklabels([movies[movies['item_id'] == idx]['title'].values[0][:30] for idx in top_20_avg_rating.index], fontsize=8)
axes[1, 1].set_xlabel('Rating Promedio')
axes[1, 1].set_title('Rating Promedio - Top 20 Películas Más Calificadas')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'exploratory_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualización guardada")

---

## Parte 2: Análisis de Sparsidad

### El Problema de la Sparsidad

**Sparsity** es el porcentaje de celdas vacías en la matriz de usuario-película.

**Fórmula:**
$$\text{Sparsity} = \frac{\text{Celdas sin rating}}{\text{Total de celdas posibles}} \times 100\%$$

**Por qué importa:**
- A mayor sparsity, menos información tenemos sobre las preferencias de los usuarios
- El reto principal del sistema de recomendación es **predecir las celdas vacías**
- La mayoría de los usuarios solo han visto una pequeña fracción de películas disponibles

In [ ]:
print("=" * 70)
print("🔍 ANÁLISIS DE SPARSITY - MOVIELENS 100K")
print("=" * 70)

# Crear la matriz de ratings (usuario-item)
# Filas = usuarios, Columnas = películas, Valores = ratings
ratings_matrix = ratings.pivot(
    index='user_id',
    columns='item_id',
    values='rating'
)

print(f"\n📐 Dimensiones de la matriz:")
print(f"   - Usuarios (filas): {ratings_matrix.shape[0]}")
print(f"   - Películas (columnas): {ratings_matrix.shape[1]}")
print(f"   - Total de celdas: {ratings_matrix.shape[0] * ratings_matrix.shape[1]:,}")

print(f"\n📊 Muestra de la matriz (primeros 5 usuarios x 5 películas):")
print(ratings_matrix.iloc[:5, :5])
print("\nNota: NaN significa que el usuario NO ha calificado esa película")

In [ ]:
# Calcular sparsity
total_cells = ratings_matrix.shape[0] * ratings_matrix.shape[1]
filled_cells = ratings_matrix.notna().sum().sum()
empty_cells = ratings_matrix.isna().sum().sum()
sparsity = (empty_cells / total_cells) * 100

print("\n" + "=" * 70)
print("🕳️  CÁLCULO DE SPARSITY")
print("=" * 70)

print(f"\n📊 Estadísticas de la matriz:")
print(f"   - Total de celdas posibles: {total_cells:,}")
print(f"   - Celdas con ratings: {filled_cells:,} ({(filled_cells/total_cells)*100:.2f}%)")
print(f"   - Celdas vacías (sin rating): {empty_cells:,} ({sparsity:.2f}%)")
print(f"\n🎯 SPARSITY: {sparsity:.2f}%")

print(f"\n💡 Interpretación:")
print(f"   - Solo el {(filled_cells/total_cells)*100:.2f}% de las interacciones posibles existen.")
print(f"   - La matriz está {sparsity:.2f}% vacía.")
print(f"   - Cada usuario solo ha visto ~{(filled_cells/ratings_matrix.shape[0]):.0f} películas")
print(f"     de las {ratings_matrix.shape[1]:,} disponibles.")

In [ ]:
# Visualizar sparsity
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análisis de Sparsity - MovieLens 100K', fontsize=16, fontweight='bold')

# 1. Visualización de la matriz (muestra de 50x50)
ax1 = axes[0, 0]
sample_matrix = ratings_matrix.iloc[:50, :50].notna().astype(int)
im = ax1.imshow(sample_matrix, cmap='RdYlGn', aspect='auto', interpolation='nearest')
ax1.set_xlabel('Películas (IDs)')
ax1.set_ylabel('Usuarios (IDs)')
ax1.set_title('Matriz de Ratings (50x50 muestra)\nVerde=Rating existe, Rojo=Sin rating')
plt.colorbar(im, ax=ax1, label='1=Rating, 0=Vacío')

# 2. Distribución de ratings por usuario
ax2 = axes[0, 1]
ratings_per_user_sparse = ratings_matrix.notna().sum(axis=1)
ax2.hist(ratings_per_user_sparse, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(ratings_per_user_sparse.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Media: {ratings_per_user_sparse.mean():.1f}')
ax2.axvline(ratings_per_user_sparse.median(), color='orange', linestyle='--', 
            linewidth=2, label=f'Mediana: {ratings_per_user_sparse.median():.1f}')
ax2.set_xlabel('Número de películas calificadas')
ax2.set_ylabel('Número de usuarios')
ax2.set_title('Distribución de Actividad por Usuario')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# 3. Distribución de ratings por película
ax3 = axes[1, 0]
ratings_per_movie_sparse = ratings_matrix.notna().sum(axis=0)
ax3.hist(ratings_per_movie_sparse, bins=50, color='coral', edgecolor='black', alpha=0.7)
ax3.axvline(ratings_per_movie_sparse.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Media: {ratings_per_movie_sparse.mean():.1f}')
ax3.axvline(ratings_per_movie_sparse.median(), color='orange', linestyle='--', 
            linewidth=2, label=f'Mediana: {ratings_per_movie_sparse.median():.1f}')
ax3.set_xlabel('Número de usuarios que calificaron')
ax3.set_ylabel('Número de películas')
ax3.set_title('Distribución de Popularidad por Película')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Gráfico de sparsity general
ax4 = axes[1, 1]
categories = ['Ratings\nExistentes', 'Celdas\nVacías']
values = [filled_cells, empty_cells]
colors = ['#2ecc71', '#e74c3c']
bars = ax4.bar(categories, values, color=colors, edgecolor='black', linewidth=2)
ax4.set_ylabel('Número de celdas')
ax4.set_title(f'Sparsity General: {sparsity:.2f}%')
ax4.set_ylim(0, total_cells * 1.1)

# Añadir etiquetas con porcentajes
for bar, value in zip(bars, values):
    height = bar.get_height()
    percentage = (value / total_cells) * 100
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{value:,}\n({percentage:.1f}%)',
             ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'sparsity_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualización guardada")

---

## Parte 3: User-Based Collaborative Filtering

### Concepto: "Si dos usuarios tienen gustos similares, probablemente les gustarán las mismas películas"

**Algoritmo:**
1. Calcular **similitud del coseno** entre todos los usuarios
2. Para recomendar, encontrar usuarios similares al usuario objetivo
3. Ver qué películas les gustaron a esos usuarios similares
4. Predecir rating usando **promedio ponderado** por similitud

**Fórmula de Predicción:**
$$\hat{r}_{u,i} = \frac{\sum_{v \in S} sim(u,v) \cdot r_{v,i}}{\sum_{v \in S} sim(u,v)}$$

Donde:
- $u$ = usuario objetivo
- $i$ = película a predecir
- $v$ = usuarios similares
- $sim(u,v)$ = similitud entre usuarios
- $r_{v,i}$ = rating del usuario similar para la película

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 70)
print("👥 USER-BASED COLLABORATIVE FILTERING")
print("=" * 70)

# Rellenar NaN con 0 para el cálculo de similitud
# Nota: esto es una simplificación, hay métodos más sofisticados
ratings_matrix_filled = ratings_matrix.fillna(0)

print("\n⚙️  Calculando matriz de similitud del coseno...")
print("   (Esto puede tardar unos segundos...)")

# Calcular similitud del coseno entre usuarios
# Cada fila es un usuario, cada columna es una película
user_similarity = cosine_similarity(ratings_matrix_filled)

# Convertir a DataFrame para facilitar el acceso
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=ratings_matrix.index,
    columns=ratings_matrix.index
)

print(f"✅ Matriz de similitud creada: {user_similarity_df.shape[0]} × {user_similarity_df.shape[1]}")

# Mostrar ejemplo de similitudes
print("\n📊 Ejemplo - Similitud del Usuario 1 con otros usuarios:")
user_1_similarities = user_similarity_df.loc[1].sort_values(ascending=False).head(6)
print(user_1_similarities)
print("\nNota: El usuario 1 tiene similitud 1.0 consigo mismo (es idéntico a sí mismo)")

In [ ]:
# Funciones para User-Based Collaborative Filtering

def find_similar_users(user_id, similarity_df, k=10):
    """
    Encuentra los k usuarios más similares a un usuario dado
    
    Args:
        user_id: ID del usuario
        similarity_df: DataFrame con similitudes
        k: Número de usuarios similares a retornar
    
    Returns:
        Series con los k usuarios más similares y sus similitudes
    """
    # Obtener similitudes del usuario
    similarities = similarity_df.loc[user_id]
    
    # Ordenar de mayor a menor (excluir el mismo usuario)
    similar_users = similarities.sort_values(ascending=False)[1:k+1]
    
    return similar_users


def predict_rating(user_id, item_id, ratings_matrix, similarity_df, k=10):
    """
    Predice el rating que un usuario daría a una película
    usando el promedio ponderado de usuarios similares
    
    Args:
        user_id: ID del usuario
        item_id: ID de la película
        ratings_matrix: Matriz de ratings
        similarity_df: Matriz de similitudes
        k: Número de usuarios similares a considerar
    
    Returns:
        Rating predicho (float)
    """
    # Encontrar usuarios similares
    similar_users = find_similar_users(user_id, similarity_df, k)
    
    # Obtener ratings de usuarios similares para esta película
    similar_users_ratings = ratings_matrix.loc[similar_users.index, item_id]
    
    # Eliminar NaN (usuarios que no han calificado esta película)
    valid_ratings = similar_users_ratings.dropna()
    valid_similarities = similar_users.loc[valid_ratings.index]
    
    # Si no hay usuarios similares que hayan calificado esta película
    if len(valid_ratings) == 0:
        # Retornar el rating promedio del usuario
        user_mean = ratings_matrix.loc[user_id].mean()
        return user_mean if not np.isnan(user_mean) else 3.0
    
    # Calcular predicción como promedio ponderado
    # rating_predicho = sum(similitud × rating) / sum(similitud)
    weighted_sum = (valid_similarities * valid_ratings).sum()
    similarity_sum = valid_similarities.sum()
    
    predicted_rating = weighted_sum / similarity_sum if similarity_sum > 0 else 3.0
    
    return predicted_rating


def recommend_movies(user_id, ratings_matrix, similarity_df, movies_df, k_users=10, n_recommendations=10):
    """
    Recomienda las top N películas para un usuario
    
    Args:
        user_id: ID del usuario
        ratings_matrix: Matriz de ratings
        similarity_df: Matriz de similitudes
        movies_df: DataFrame con información de películas
        k_users: Número de usuarios similares a considerar
        n_recommendations: Número de recomendaciones a retornar
    
    Returns:
        DataFrame con recomendaciones
    """
    # Obtener películas que el usuario YA ha visto
    user_ratings = ratings_matrix.loc[user_id]
    seen_movies = user_ratings.dropna().index.tolist()
    
    # Obtener películas que NO ha visto
    all_movies = ratings_matrix.columns.tolist()
    unseen_movies = [movie for movie in all_movies if movie not in seen_movies]
    
    print(f"\n🎬 Usuario {user_id}:")
    print(f"   - Ha visto: {len(seen_movies)} películas")
    print(f"   - No ha visto: {len(unseen_movies)} películas")
    print(f"   - Calculando predicciones para películas no vistas...")
    
    # Predecir ratings para películas no vistas
    predictions = []
    for movie_id in unseen_movies:
        pred_rating = predict_rating(user_id, movie_id, ratings_matrix, similarity_df, k_users)
        predictions.append({
            'item_id': movie_id,
            'predicted_rating': pred_rating
        })
    
    # Convertir a DataFrame y ordenar
    predictions_df = pd.DataFrame(predictions)
    predictions_df = predictions_df.sort_values('predicted_rating', ascending=False)
    
    # Obtener top N recomendaciones
    top_recommendations = predictions_df.head(n_recommendations)
    
    # Añadir información de películas
    top_recommendations = top_recommendations.merge(
        movies_df[['item_id', 'title']],
        on='item_id',
        how='left'
    )
    
    return top_recommendations

print("✅ Funciones definidas")

### Probando User-Based Collaborative Filtering

In [ ]:
# Seleccionar un usuario de ejemplo
test_user_id = 1

print("\n" + "=" * 70)
print("🎯 PROBANDO EL SISTEMA DE RECOMENDACIÓN")
print("=" * 70)

print(f"\n👤 Usuario de prueba: {test_user_id}")

# Ver las películas que ya ha calificado
user_ratings = ratings_matrix.loc[test_user_id].dropna().sort_values(ascending=False)
user_movies = user_ratings.head(10)

print(f"\n⭐ Top 10 películas que el Usuario {test_user_id} ya calificó alto:")
for item_id, rating in user_movies.items():
    movie_title = movies[movies['item_id'] == item_id]['title'].values[0]
    print(f"   {rating:.0f} ⭐ - {movie_title}")

In [ ]:
# Obtener recomendaciones usando User-Based CF
recommendations_user_based = recommend_movies(
    user_id=test_user_id,
    ratings_matrix=ratings_matrix,
    similarity_df=user_similarity_df,
    movies_df=movies,
    k_users=20,
    n_recommendations=10
)

print(f"\n🎬 Top 10 Recomendaciones para el Usuario {test_user_id}:")
print("=" * 70)
for idx, row in recommendations_user_based.iterrows():
    print(f"{row['predicted_rating']:.2f} ⭐ - {row['title']}")

In [ ]:
# Visualizar similitudes entre usuarios
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('User-Based Collaborative Filtering - Análisis de Similitud', 
             fontsize=14, fontweight='bold')

# 1. Heatmap de similitud (muestra 30x30)
ax1 = axes[0]
sample_similarity = user_similarity_df.iloc[:30, :30]
sns.heatmap(sample_similarity, cmap='RdYlGn', center=0.5, 
            square=True, ax=ax1, cbar_kws={'label': 'Similitud'})
ax1.set_title('Matriz de Similitud entre Usuarios (muestra 30×30)')
ax1.set_xlabel('Usuario ID')
ax1.set_ylabel('Usuario ID')

# 2. Distribución de similitudes
ax2 = axes[1]
# Obtener todas las similitudes (excluyendo diagonal)
all_similarities = user_similarity_df.values[np.triu_indices_from(user_similarity_df.values, k=1)]
ax2.hist(all_similarities, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(all_similarities.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Media: {all_similarities.mean():.3f}')
ax2.set_xlabel('Similitud del Coseno')
ax2.set_ylabel('Frecuencia')
ax2.set_title('Distribución de Similitudes entre Usuarios')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'user_based_cf.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualización guardada")

---

## Parte 4: Item-Based Collaborative Filtering

### Concepto: "Si a un usuario le gustó una película, probablemente le gustará una película similar"

**Ventajas sobre User-Based:**
- Las películas no cambian, pero los usuarios sí → Más estable
- Valores de similitud más altos → Mejores predicciones
- Más fácil de explicar: "Porque te gustó X, te recomendamos Y"

**Algoritmo:**
1. Transponer la matriz (películas se convierten en filas)
2. Calcular similitud entre películas
3. Para cada película no vista, encontrar películas similares que el usuario YA calificó
4. Predecir rating usando promedio ponderado de películas similares

In [ ]:
print("=" * 70)
print("🎬 ITEM-BASED COLLABORATIVE FILTERING")
print("=" * 70)

# Transponer matriz para calcular similitud entre películas
ratings_matrix_filled = ratings_matrix.fillna(0)
ratings_matrix_T = ratings_matrix_filled.T  # Transponer

print("\n⚙️  Calculando matriz de similitud entre películas...")
print("   (Esto puede tardar unos segundos...)")

# Calcular similitud del coseno entre películas
item_similarity = cosine_similarity(ratings_matrix_T)

# Convertir a DataFrame
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=ratings_matrix.columns,
    columns=ratings_matrix.columns
)

print(f"✅ Matriz de similitud de películas creada: {item_similarity_df.shape[0]} × {item_similarity_df.shape[1]}")

# Ejemplo: película similar a la película 1
movie_id_example = 1
movie_title_example = movies[movies['item_id'] == movie_id_example]['title'].values[0]
print(f"\n📊 Películas más similares a '{movie_title_example}':")
similar_movies = item_similarity_df.loc[movie_id_example].sort_values(ascending=False).head(6)
for idx, (movie_id, similarity) in enumerate(similar_movies.items()):
    movie_title = movies[movies['item_id'] == movie_id]['title'].values[0]
    print(f"{idx+1}. {movie_title} (similitud: {similarity:.3f})")

In [ ]:
# Funciones para Item-Based Collaborative Filtering

def find_similar_items(item_id, similarity_df, k=10):
    """
    Encuentra las k películas más similares a una película dada
    """
    similarities = similarity_df.loc[item_id]
    similar_items = similarities.sort_values(ascending=False)[1:k+1]
    return similar_items


def predict_rating_item_based(user_id, item_id, ratings_matrix, similarity_df, k=10):
    """
    Predice el rating basado en similitud entre películas
    """
    # Encontrar películas similares
    similar_items = find_similar_items(item_id, similarity_df, k)
    
    # Obtener ratings del usuario para películas similares
    similar_items_ratings = ratings_matrix.loc[user_id, similar_items.index]
    
    # Eliminar NaN
    valid_ratings = similar_items_ratings.dropna()
    valid_similarities = similar_items.loc[valid_ratings.index]
    
    if len(valid_ratings) == 0:
        return ratings_matrix.loc[user_id].mean() if ratings_matrix.loc[user_id].notna().any() else 3.0
    
    # Calcular promedio ponderado
    weighted_sum = (valid_similarities * valid_ratings).sum()
    similarity_sum = valid_similarities.sum()
    
    return weighted_sum / similarity_sum if similarity_sum > 0 else 3.0


def recommend_movies_item_based(user_id, ratings_matrix, similarity_df, movies_df, k_items=10, n_recommendations=10):
    """
    Recomienda películas basándose en similitud de películas
    """
    user_ratings = ratings_matrix.loc[user_id]
    seen_movies = user_ratings.dropna().index.tolist()
    
    all_movies = ratings_matrix.columns.tolist()
    unseen_movies = [movie for movie in all_movies if movie not in seen_movies]
    
    print(f"\n🎬 Usuario {user_id} (Item-Based):")
    print(f"   - Ha visto: {len(seen_movies)} películas")
    print(f"   - No ha visto: {len(unseen_movies)} películas")
    print(f"   - Calculando predicciones...")
    
    # Predecir ratings
    predictions = []
    for movie_id in unseen_movies:
        pred_rating = predict_rating_item_based(user_id, movie_id, ratings_matrix, similarity_df, k_items)
        predictions.append({
            'item_id': movie_id,
            'predicted_rating': pred_rating
        })
    
    predictions_df = pd.DataFrame(predictions)
    predictions_df = predictions_df.sort_values('predicted_rating', ascending=False)
    top_recommendations = predictions_df.head(n_recommendations)
    
    top_recommendations = top_recommendations.merge(
        movies_df[['item_id', 'title']],
        on='item_id',
        how='left'
    )
    
    return top_recommendations

print("✅ Funciones definidas")

In [ ]:
# Obtener recomendaciones usando Item-Based CF
recommendations_item_based = recommend_movies_item_based(
    user_id=test_user_id,
    ratings_matrix=ratings_matrix,
    similarity_df=item_similarity_df,
    movies_df=movies,
    k_items=20,
    n_recommendations=10
)

print(f"\n🎬 Top 10 Recomendaciones (Item-Based) para Usuario {test_user_id}:")
print("=" * 70)
for idx, row in recommendations_item_based.iterrows():
    print(f"{row['predicted_rating']:.2f} ⭐ - {row['title']}")

---

## Parte 5: Matrix Factorization (SVD)

### Concepto: "Descomponer la matriz de ratings en factores latentes"

**Idea Principal:**
- Las películas pueden representarse por características latentes (géneros, directores, etc.) 
- Los usuarios pueden representarse por preferencias latentes
- Los ratings son producto de estas preferencias latentes

**Descomposición SVD:**
$$R \approx U \times \Sigma \times V^T$$

Donde:
- **U**: Matriz de usuarios × factores latentes (943 × k)
- **Σ**: Matriz diagonal de importancia de factores (k × k)  
- **V^T**: Matriz de factores latentes × películas (k × 1,682)

**Ventajas:**
- Captura información latente no visible
- Generalmente mejor accuracy que Collaborative Filtering
- Escalable para sistemas grandes

**Desventajas:**
- Menos interpretable que item/user-based
- Difícil de explicar por qué se hace una recomendación

In [ ]:
from scipy.sparse.linalg import svds

print("=" * 70)
print("🔢 MATRIX FACTORIZATION (SVD)")
print("=" * 70)

# Normalizar la matriz restando el promedio por usuario
print("\n⚙️  Normalizando matriz...")
user_mean = ratings_matrix_filled.mean(axis=1)
ratings_matrix_normalized = ratings_matrix_filled.sub(user_mean, axis=0)

# Aplicar SVD
print("\n⚙️  Aplicando Singular Value Decomposition (SVD)...")
k = 50  # Número de factores latentes
print(f"   Reduciendo {ratings_matrix_normalized.shape[1]} dimensiones a {k} factores latentes...")

U, sigma, Vt = svds(ratings_matrix_normalized, k=k)

print(f"\n✅ SVD completado:")
print(f"   - U (usuarios × factores): {U.shape}")
print(f"   - Σ (valores singulares): {len(sigma)}")
print(f"   - V^T (factores × películas): {Vt.shape}")

# Reconstruir matriz de predicciones
print("\n⚙️  Reconstruyendo matriz de predicciones...")
sigma_matrix = np.diag(sigma)
predictions_svd = np.dot(np.dot(U, sigma_matrix), Vt)

# Desnormalizar (añadir el promedio de usuario de vuelta)
predictions_svd = predictions_svd + user_mean.values.reshape(-1, 1)

# Clip a rango [1, 5]
predictions_svd = np.clip(predictions_svd, 1, 5)

# Convertir a DataFrame
predictions_df_svd = pd.DataFrame(
    predictions_svd,
    index=ratings_matrix.index,
    columns=ratings_matrix.columns
)

print(f"✅ Matriz de predicciones: {predictions_df_svd.shape}")

In [ ]:
# Función para recomendar basada en SVD
def recommend_movies_svd(user_id, predictions_df, ratings_matrix, movies_df, n_recommendations=10):
    """
    Recomienda películas usando predicciones de SVD
    """
    # Obtener películas vistas
    user_ratings = ratings_matrix.loc[user_id]
    seen_movies = user_ratings.dropna().index.tolist()
    
    # Obtener predicciones para películas no vistas
    predictions = predictions_df.loc[user_id]
    unseen_predictions = predictions.drop(seen_movies, errors='ignore')
    
    # Ordenar y obtener top N
    top_predictions = unseen_predictions.nlargest(n_recommendations)
    
    # Convertir a DataFrame
    recommendations = pd.DataFrame({
        'item_id': top_predictions.index,
        'predicted_rating': top_predictions.values
    }).reset_index(drop=True)
    
    # Añadir información de películas
    recommendations = recommendations.merge(
        movies_df[['item_id', 'title']],
        on='item_id',
        how='left'
    )
    
    return recommendations

# Obtener recomendaciones SVD
recommendations_svd = recommend_movies_svd(
    user_id=test_user_id,
    predictions_df=predictions_df_svd,
    ratings_matrix=ratings_matrix,
    movies_df=movies,
    n_recommendations=10
)

print(f"\n🎬 Top 10 Recomendaciones (SVD) para Usuario {test_user_id}:")
print("=" * 70)
for idx, row in recommendations_svd.iterrows():
    print(f"{row['predicted_rating']:.2f} ⭐ - {row['title']}")

In [ ]:
# Visualizar análisis de SVD
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Matrix Factorization (SVD) - Análisis', fontsize=16, fontweight='bold')

# 1. Valores singulares (importancia de cada factor)
ax1 = axes[0, 0]
ax1.plot(range(1, len(sigma) + 1), np.sort(sigma)[::-1], 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Factor Latente')
ax1.set_ylabel('Valor Singular (σ)')
ax1.set_title('Importancia de Factores Latentes')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 2. Varianza acumulativa explicada
ax2 = axes[0, 1]
cumulative_variance = np.cumsum(sigma[::-1]) / np.sum(sigma)
ax2.plot(range(1, len(sigma) + 1), cumulative_variance, 'g-o', linewidth=2, markersize=6)
ax2.axhline(y=0.9, color='r', linestyle='--', label='90% varianza')
ax2.set_xlabel('Número de Factores')
ax2.set_ylabel('Varianza Acumulativa Explicada')
ax2.set_title('Varianza Explicada vs Número de Factores')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Error de predicción vs película real
ax3 = axes[1, 0]
errors = []
for movie_id in ratings_matrix.columns:
    mask = ratings_matrix[movie_id].notna()
    if mask.sum() > 0:
        real_ratings = ratings_matrix.loc[mask, movie_id]
        pred_ratings = predictions_df_svd.loc[mask, movie_id]
        rmse = np.sqrt(np.mean((real_ratings - pred_ratings) ** 2))
        errors.append(rmse)

ax3.hist(errors, bins=50, color='coral', edgecolor='black', alpha=0.7)
ax3.axvline(np.mean(errors), color='red', linestyle='--', linewidth=2, label=f'RMSE medio: {np.mean(errors):.3f}')
ax3.set_xlabel('RMSE')
ax3.set_ylabel('Número de películas')
ax3.set_title('Distribución del Error de Predicción')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Real vs Predicho (muestra)
ax4 = axes[1, 1]
sample_size = 100
sample_real = []
sample_pred = []
for i in range(min(sample_size, len(ratings))):
    user_id = ratings.iloc[i]['user_id']
    item_id = ratings.iloc[i]['item_id']
    real_rating = ratings_matrix.loc[user_id, item_id]
    pred_rating = predictions_df_svd.loc[user_id, item_id]
    sample_real.append(real_rating)
    sample_pred.append(pred_rating)

ax4.scatter(sample_real, sample_pred, alpha=0.6, s=50)
ax4.plot([1, 5], [1, 5], 'r--', linewidth=2, label='Predicción Perfecta')
ax4.set_xlabel('Rating Real')
ax4.set_ylabel('Rating Predicho')
ax4.set_title('Ratings Reales vs Predichos (muestra)')
ax4.set_xlim(0.5, 5.5)
ax4.set_ylim(0.5, 5.5)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'svd_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualización guardada")

---

## Parte 6: Sistema Híbrido

### Combinando Múltiples Enfoques

La idea es combinar lo mejor de cada método:
- **SVD**: Buenas predicciones numéricas
- **Item-Based**: Explicabilidad ("porque te gustó X")
- **User-Based**: Captura gustos de usuarios similares

**Estrategia:**
1. Usar SVD para obtener predicciones numéricas de calidad
2. Usar Item-Based para encontrar películas similares
3. Combinar scores y proporcionar explicaciones

In [ ]:
def get_svd_recommendations(user_id, predictions_df, ratings_matrix, movies_df, n=10):
    """
    Obtiene recomendaciones del modelo SVD
    """
    user_ratings = ratings_matrix.loc[user_id]
    seen_movies = user_ratings.dropna().index.tolist()
    
    predictions = predictions_df.loc[user_id]
    unseen_predictions = predictions.drop(seen_movies, errors='ignore')
    top_predictions = unseen_predictions.nlargest(n)
    
    return top_predictions


def get_item_based_recommendation_reason(user_id, item_id, ratings_matrix, item_similarity_df, movies_df, top_k=3):
    """
    Obtiene una explicación para una recomendación usando Item-Based CF
    Busca películas similares que el usuario ya calificó alto
    """
    user_ratings = ratings_matrix.loc[user_id].dropna()
    
    # Obtener películas similares a la recomendación
    similar_movies = item_similarity_df.loc[item_id].sort_values(ascending=False)[1:top_k+1]
    
    # Ver cuáles el usuario ya calificó
    reasons = []
    for sim_movie_id, similarity in similar_movies.items():
        if sim_movie_id in user_ratings.index:
            user_rating = user_ratings.loc[sim_movie_id]
            movie_title = movies_df[movies_df['item_id'] == sim_movie_id]['title'].values[0]
            reasons.append(f"Te gustó '{movie_title}' ({user_rating:.0f}⭐)")
    
    return reasons


def hybrid_recommendations(user_id, predictions_df_svd, ratings_matrix, 
                          item_similarity_df, movies_df, n_recommendations=10):
    """
    Sistema híbrido que combina SVD + Item-Based
    """
    # Obtener recomendaciones SVD
    svd_recs = get_svd_recommendations(user_id, predictions_df_svd, ratings_matrix, movies_df, n_recommendations)
    
    # Para cada recomendación, obtener explicación
    recommendations = []
    for item_id in svd_recs.index:
        movie_title = movies_df[movies_df['item_id'] == item_id]['title'].values[0]
        pred_rating = svd_recs.loc[item_id]
        reasons = get_item_based_recommendation_reason(user_id, item_id, ratings_matrix, 
                                                       item_similarity_df, movies_df, top_k=3)
        
        recommendations.append({
            'item_id': item_id,
            'title': movie_title,
            'predicted_rating': pred_rating,
            'reason': reasons[0] if reasons else "Basado en patrones similares a tu perfil"
        })
    
    return recommendations


print("✅ Funciones del sistema híbrido definidas")

In [ ]:
# Obtener recomendaciones híbridas
print("\n" + "=" * 70)
print("🎬 SISTEMA HÍBRIDO DE RECOMENDACIONES")
print("=" * 70)

hybrid_recs = hybrid_recommendations(
    user_id=test_user_id,
    predictions_df_svd=predictions_df_svd,
    ratings_matrix=ratings_matrix,
    item_similarity_df=item_similarity_df,
    movies_df=movies,
    n_recommendations=10
)

print(f"\n🎬 Top 10 Recomendaciones HÍBRIDAS para Usuario {test_user_id}:")
print("=" * 70)
for idx, rec in enumerate(hybrid_recs, 1):
    print(f"\n{idx}. {rec['title']}")
    print(f"   Rating predicho: {rec['predicted_rating']:.2f} ⭐")
    print(f"   Razón: {rec['reason']}")

---

## Resumen y Conclusiones

### Comparación de Métodos

| Método | Ventajas | Desventajas |
|--------|----------|-------------|
| **User-Based CF** | Simple, explicable | Sensible a sparsity, lento con muchos usuarios |
| **Item-Based CF** | Estable, explicable | Requiere similitud entre items |
| **SVD / Matrix Fact.** | Alta precisión, escalable | Poco explicable, difícil de interpretar |
| **Híbrido** | Combina precisión + explicabilidad | Más complejo computacionalmente |

### Problemas Clave en Sistemas de Recomendación

1. **Cold Start Problem**
   - Usuarios nuevos (sin ratings): No hay datos históricos
   - Películas nuevas: Pocos usuarios las han visto
   - Solución: Usar metadatos, información demográfica, o regresión

2. **Sparsity Problem**
   - 93.7% de la matriz está vacía
   - Los usuarios ven solo una fracción de películas
   - Solución: Factorización de matriz, regularización

3. **Serendipity (Descubrimiento)**
   - Los sistemas tienden a recomendar lo "obvio"
   - Los usuarios quieren sorpresas, no películas similares a las que ya vieron
   - Solución: Mezclar recomendaciones con diversidad

### Métricas de Evaluación (Producción)

```python
# Precisión y Recall
Precision@K = (# items recomendados relevantes) / K
Recall@K = (# items recomendados relevantes) / (# items relevantes totales)

# RMSE (para predicción de ratings)
RMSE = sqrt(mean((real - predicho)^2))

# NDCG (Normalized Discounted Cumulative Gain)
# Evalúa ordenamiento de recomendaciones
```

In [ ]:
# Crear visualización comparativa de métodos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Comparación de Métodos de Recomendación - Usuario {test_user_id}', fontsize=14, fontweight='bold')

# Datos para comparación
user_based_ratings = recommendations_user_based['predicted_rating'].values[:5]
item_based_ratings = recommendations_item_based['predicted_rating'].values[:5]
svd_ratings = recommendations_svd['predicted_rating'].values[:5]

titles_ub = recommendations_user_based['title'].values[:5]
titles_ib = recommendations_item_based['title'].values[:5]
titles_svd = recommendations_svd['title'].values[:5]

# 1. User-Based
axes[0].barh(range(len(user_based_ratings)), user_based_ratings, color='skyblue', edgecolor='black')
axes[0].set_yticks(range(len(user_based_ratings)))
axes[0].set_yticklabels([t[:25] for t in titles_ub], fontsize=9)
axes[0].set_xlabel('Rating Predicho')
axes[0].set_title('User-Based CF')
axes[0].set_xlim(0, 5.5)
axes[0].grid(axis='x', alpha=0.3)

# 2. Item-Based
axes[1].barh(range(len(item_based_ratings)), item_based_ratings, color='lightcoral', edgecolor='black')
axes[1].set_yticks(range(len(item_based_ratings)))
axes[1].set_yticklabels([t[:25] for t in titles_ib], fontsize=9)
axes[1].set_xlabel('Rating Predicho')
axes[1].set_title('Item-Based CF')
axes[1].set_xlim(0, 5.5)
axes[1].grid(axis='x', alpha=0.3)

# 3. SVD
axes[2].barh(range(len(svd_ratings)), svd_ratings, color='lightgreen', edgecolor='black')
axes[2].set_yticks(range(len(svd_ratings)))
axes[2].set_yticklabels([t[:25] for t in titles_svd], fontsize=9)
axes[2].set_xlabel('Rating Predicho')
axes[2].set_title('Matrix Factorization (SVD)')
axes[2].set_xlim(0, 5.5)
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'comparison_methods.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Comparación guardada")

### Conclusiones

En este notebook hemos explorado:

1. **Análisis de datos**: Entendimiento del dataset MovieLens 100K
2. **Sparsity**: El problema central de sistemas de recomendación
3. **User-Based CF**: Encontrar usuarios similares
4. **Item-Based CF**: Encontrar películas similares
5. **Matrix Factorization**: Descomposición latente para predicciones de calidad
6. **Sistemas Híbridos**: Combinación de métodos para mejores resultados

**Próximos pasos en producción:**
- Implementar evaluación cruzada (cross-validation)
- Medir precisión, recall, NDCG
- Optimizar hiperparámetros (k, número de factores)
- Servir modelo en API (FastAPI, Flask)
- Monitorear recomendaciones en tiempo real
- A/B testing con usuarios reales

**Para más información:**
- Ver `amazone/src/` para implementaciones completas
- Ejecutar `python main.py` desde la carpeta amazone
- Probar la aplicación web: `streamlit run web/app.py`